# Estrazione della lunghezza d'onda da file FITS

I file FITS possono contenere diversi tipi di dati astronomici:
- **Immagini** (2D): fotografie di oggetti celesti
- **Spettri** (1D): intensità in funzione della lunghezza d'onda
- **Cube** (3D): combinazione di immagini a diverse lunghezze d'onda

In questo notebook lavoreremo con l'immagine FITS di **M87** (un'immagine 2D nel piano celeste).

**Nota importante**: M87_NEW.fits è un'immagine astronomica (coordinate RA/DEC), non uno spettro. Le coordinate spaziali sono definite nel sistema WCS (World Coordinate System). Per dati spettrali reali, l'asse X rappresenterebbe la lunghezza d'onda. Qui mostreremo comunque come estrarre parametri dall'header e come funziona il sistema di coordinate WCS.

## 1. Import librerie e verifica file

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS

%matplotlib inline

def verifica_file(filepath: str) -> bool:
    """Verifica l'esistenza di un file."""
    esiste = os.path.exists(filepath)
    print(f"File: {filepath} - Esiste: {esiste}")
    return esiste

file_path = './M87_NEW.fits'
if not verifica_file(file_path):
    print("ATTENZIONE: File non trovato. Alcune celle potrebbero non funzionare.")

## 2. Lettura dell'header FITS

L'header di un file FITS contiene i **metadati** dell'osservazione. Le parole chiave più importanti per il sistema di coordinate sono:

### Parametri WCS (World Coordinate System)
| Parametro | Descrizione |
|-----------|-------------|
| `CRVAL1/2` | Valore della coordinata al pixel di riferimento |
| `CDELT1/2` | Incremento di coordinata per pixel (passo) |
| `CRPIX1/2` | Pixel di riferimento (indice del pixel) |
| `CTYPE1/2` | Tipo di proiezione (es. RA---TAN, DEC--TAN) |
| `EQUINOX` | Epoca dell'equinozio (es. 2000.0) |

#### Formula per le coordinate
$$
\lambda_i = \text{CRVAL1} + (i - \text{CRPIX1}) \times \text{CDELT1}
$$

Nel nostro caso, CRVAL1/CRVAL2 sono longitudine e latitudine celeste (RA/DEC in gradi), non lunghezza d'onda.

In [ ]:
def leggi_header_fits(filepath: str):
    """Legge l'header di un file FITS.

    Args:
        filepath: Percorso del file FITS

    Returns:
        Header FITS se riuscito, None altrimenti
    """
    try:
        with fits.open(filepath, memmap=False) as hdul:
            header = hdul[0].header
            print("Contenuto dell'header FITS:")
            print(repr(header))
            return header
    except Exception as e:
        print(f"ERRORE nella lettura del file: {e}")
        return None

header = leggi_header_fits(file_path)

## 3. Estrazione dei parametri WCS

Estraiamo i parametri `CRVAL1`, `CDELT1`, `CRPIX1` dall'header per capire la calibrazione spaziale dell'immagine. Per un'immagine spaziale (non spettrale), questi valori si riferiscono a coordinate RA/DEC.

In [ ]:
def estrai_parametri_wcs(header) -> dict:
    """Estrae i parametri WCS di base dall'header FITS.

    Args:
        header: Header FITS

    Returns:
        Dizionario con i parametri WCS,
        o dizionario vuoto se l'header è None
    """
    if header is None:
        return {}

    # Parole chiave da estrarre con valori predefiniti
    chiavi = ['CRVAL1', 'CRVAL2', 'CDELT1', 'CDELT2',
              'CRPIX1', 'CRPIX2', 'CTYPE1', 'CTYPE2', 'EQUINOX']

    parametri = {}
    print("Parametri WCS estratti dall'header:")
    for k in chiavi:
        try:
            parametri[k] = header[k]
            print(f"  {k:8s} = {parametri[k]}")
        except KeyError:
            parametri[k] = None
            print(f"  {k:8s} = NON TROVATO")

    return parametri

parametri = estrai_parametri_wcs(header)
if parametri:
    print(f"\nTipo proiezione: {parametri.get('CTYPE1')} / {parametri.get('CTYPE2')}")

## 4. Calcolo delle coordinate usando la formula lineare

Usiamo la formula standard per calcolare le coordinate spaziali dai pixel:

$$\text{coordinata}_i = \text{CRVAL1} + (i - \text{CRPIX1}) \times \text{CDELT1}$$

Questa è la stessa formula che si userebbe per calcolare la **lunghezza d'onda** in uno spettro, ma qui otteniamo coordinate RA/DEC in gradi.

In [ ]:
def calcola_coordinate_da_header(header, n_pixels: int = 300) -> np.ndarray:
    """Calcola le coordinate (RA/DEC) usando CRVAL1, CDELT1, CRPIX1.

    Args:
        header: Header FITS
        n_pixels: Numero di pixel lungo l'asse

    Returns:
        Array di coordinate calcolate, o array vuoto se header è None
    """
    if header is None:
        print("Header non disponibile.")
        return np.array([])

    try:
        crval1 = header['CRVAL1']
        cdelt1 = header['CDELT1']
        crpix1 = header['CRPIX1']

        pixel_indices = np.arange(1, n_pixels + 1)
        coordinate = crval1 + (pixel_indices - crpix1) * cdelt1
        return coordinate
    except KeyError as e:
        print(f"ERRORE: Chiave mancante nell'header: {e}")
        return np.array([])


coordinate = calcola_coordinate_da_header(header, n_pixels=300)
if len(coordinate) > 0:
    print(f"Prime 5 coordinate RA (gradi): {coordinate[:5]}")
    print(f"Ultime 5 coordinate RA (gradi): {coordinate[-5:]}")
    print(f"Range: [{np.min(coordinate):.4f}, {np.max(coordinate):.4f}] gradi")

## 5. Uso del World Coordinate System (WCS)

Astropy fornisce il modulo `astropy.wcs.WCS` per gestire in modo avanzato le trasformazioni pixel → coordinate del mondo reale. È molto più potente della semplice formula lineare, in quanto supporta:
- Proiezioni complesse (TAN, SIN, ZEA, ecc.)
- Distorsioni ottiche
- Sistemi di coordinate multiple

**Attenzione**: Il file M87_NEW.fits contiene coordinate **spaziali** (RA/DEC), non spettrali. Per dati spettrali reali, `CTYPE1` avrebbe valori come `WAVE` o `LAMBDA`.

In [ ]:
def usa_wcs(header, n_pixels: int = 300):
    """Usa WCS per convertire coordinate pixel → mondo.

    Args:
        header: Header FITS
        n_pixels: Numero di pixel
    """
    if header is None:
        print("Header non disponibile.")
        return

    try:
        wcs = WCS(header)
        print(f"Tipo WCS: {wcs.wcs.ctype}")

        # Converti pixel lungo l'asse X in coordinate spaziali
        pixel_coords = np.arange(n_pixels)
        world_coords = wcs.pixel_to_world(pixel_coords, np.zeros(n_pixels))

        print(f"\nCoordinate convertite (prime 5):")
        for i in range(5):
            print(f"  Pixel {pixel_coords[i]:3d} -> RA={world_coords[0][i]:.4f} deg, "
                  f"DEC={world_coords[1][i]:.4f} deg")

        return wcs, world_coords
    except Exception as e:
        print(f"ERRORE WCS: {e}")
        return None, None

wcs, world_coords = usa_wcs(header)

## 6. Visualizzazione della relazione pixel-coordinate

Visualizziamo la relazione lineare tra indice del pixel e coordinata spaziale (RA). La relazione è perfettamente lineare perché il file usa una proiezione TAN (Gnomonic) con CDELT costante.

In [ ]:
def plot_coordinate_vs_pixel(coordinate: np.ndarray, titolo: str = 'Coordinate RA vs Pixel'):
    """Grafico della relazione pixel-coordinate.

    Args:
        coordinate: Array di coordinate
        titolo: Titolo del grafico
    """
    if len(coordinate) == 0:
        print("Nessun dato da plottare.")
        return

    pixel_indices = np.arange(1, len(coordinate) + 1)

    plt.figure(figsize=(10, 6))
    plt.plot(pixel_indices, coordinate, linewidth=1.5, color='steelblue')
    plt.title(titolo, fontsize=14, fontweight='bold')
    plt.xlabel('Indice del pixel', fontsize=12)
    plt.ylabel('Coordinata RA (gradi)', fontsize=12)
    plt.grid(True, alpha=0.3)

    # Linea orizzontale al centro
    centro = coordinate[len(coordinate) // 2]
    plt.axhline(y=centro, color='crimson', linestyle='--',
                linewidth=1.5, label=f'RA centrale = {centro:.4f}°')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Calcolo della pendenza (CDELT)
    pendenza = coordinate[1] - coordinate[0]
    print(f"Andamento lineare verificato.")
    print(f"Pendenza (CDELT1): {pendenza:.8f} gradi/pixel")

plot_coordinate_vs_pixel(coordinate)

## 7. Normalizzazione delle coordinate (Min-Max)

A volte è utile normalizzare i dati in un intervallo [0, 1] per confronti o per algoritmi di machine learning. La normalizzazione Min-Max si applica con la formula:

$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

In [ ]:
def normalizza_minmax(arr: np.ndarray) -> np.ndarray:
    """Normalizza un array con il metodo Min-Max in [0, 1].

    Args:
        arr: Array da normalizzare

    Returns:
        Array normalizzato (o array vuoto se input vuoto)
    """
    if len(arr) == 0:
        return np.array([])

    min_val = np.min(arr)
    max_val = np.max(arr)

    if max_val == min_val:
        print("ATTENZIONE: max = min, restituisco array di zeri.")
        return np.zeros_like(arr)

    normalizzato = (arr - min_val) / (max_val - min_val)
    return normalizzato


coordinate_norm = normalizza_minmax(coordinate)
if len(coordinate_norm) > 0:
    print(f"Coordinate normalizzate (primi 5): {coordinate_norm[:5]}")
    print(f"Range normalizzato: [{np.min(coordinate_norm):.6f}, {np.max(coordinate_norm):.6f}]")

## 8. Grafico comparativo: dati originali vs normalizzati

In [ ]:
def plot_confronto(originale: np.ndarray, normalizzato: np.ndarray):
    """Confronto grafico tra dati originali e normalizzati.

    Args:
        originale: Array originale
        normalizzato: Array normalizzato in [0, 1]
    """
    if len(originale) == 0 or len(normalizzato) == 0:
        print("Dati insufficienti per il confronto.")
        return

    pixel_indices = np.arange(1, len(originale) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.plot(pixel_indices, originale, color='steelblue', linewidth=1.5)
    ax1.set_title('Coordinate originali (gradi)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Indice del pixel')
    ax1.set_ylabel('RA (gradi)')
    ax1.grid(True, alpha=0.3)

    ax2.plot(pixel_indices, normalizzato, color='coral', linewidth=1.5)
    ax2.set_title('Coordinate normalizzate [0, 1]', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Indice del pixel')
    ax2.set_ylabel('RA normalizzata')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_confronto(coordinate, coordinate_norm)

## Riepilogo

In questo notebook abbiamo:
- Esplorato l'header FITS e i parametri WCS (CRVAL/CDELT/CRPIX/CTYPE)
- Calcolato coordinate spaziali usando la formula lineare standard
- Utilizzato `astropy.wcs.WCS` per la conversione pixel → coordinate
- Visualizzato la relazione tra pixel e coordinate
- Applicato la normalizzazione Min-Max ai dati

**Per dati spettrali reali**: Se il file FITS contenesse uno spettro (CTYPE1='WAVE'), la stessa procedura produrrebbe direttamente le lunghezze d'onda in nanometri o ångström. Il file di M87 usato qui è un'immagine spaziale, non spettrale.